In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import mysql.connector
import os

In [ ]:
load_dotenv()

connection = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = connection.cursor()

In [ ]:
# Univariate analysis
query ="""
SELECT order_id,
SUM(price + freight_value) AS total_order_value
FROM order_items
GROUP BY order_id;
"""


df_order_value = pd.read_sql(query, connection)

df_order_value.head()

df_order_value['total_order_value'].describe()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df_order_value,
    x="total_order_value",
    bins=50,
    kde=True
)

plt.title("Distribution of Total Order Value")
plt.xlabel("Total Order Value")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    data=df_order_value,
    x="total_order_value"
)

plt.title("Box plot of Total Order Value")
plt.xlabel("Total Order Value")
plt.tight_layout()
plt.show()


In [ ]:
# Bivariate analysis :- Examine relationships between two variables.
# Comparing order and order value


query ="""
SELECT o.order_status,
oi.order_id,
SUM(oi.price + oi.freight_value) AS total_order_value
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY o.order_status, oi.order_id;
"""

df_status_value = pd.read_sql(query,connection)

plt.figure(figsize=(10,5))
sns.boxplot(
    data=df_status_value,
    x='order_status',
    y ="total_order_value"
)

plt.title('Order Value by Order Status')
plt.xlabel("Order Status")
plt.ylabel("Total Order Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Analyzing order value by product category

query = """
SELECT
COALESCE(
pct.product_category_name_english,
 p.product_category_name,
'Unknown'
) AS product_category,
ROUND(SUM(oi.price + oi.freight_value), 2) AS category_revenue
FROM order_items oi
JOIN products p
ON oi.product_id = p.product_id
LEFT JOIN product_category_translation pct
ON p.product_category_name = pct.product_category_name
GROUP BY product_category
ORDER BY category_revenue DESC;
"""

df_category_revenue = pd.read_sql(query, connection)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=df_category_revenue.head(10),
    x="category_revenue",
    y="product_category"
)

plt.title("Top 10 Product Categories by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()


In [ ]:
# Trend analysis
# Analyze how order volume and revenue change over time.

query = """
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m') AS order_month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS total_revenue
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY order_month
ORDER BY order_month;
"""

df_monthly_trends = pd.read_sql(query, connection)

df_monthly_trends.head()

plt.figure(figsize=(14, 6))

sns.lineplot(
    data=df_monthly_trends,
    x="order_month",
    y="total_revenue",
    marker="o"
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
# Check whether order value and freight cost are related.

query = """
SELECT
    order_id,
    SUM(price) AS product_value,
    SUM(freight_value) AS freight_value
FROM order_items
GROUP BY order_id;
"""

df_correlation = pd.read_sql(query, connection)

df_correlation[["product_value", "freight_value"]].corr()

In [ ]:
'''
Near +1: strong positive relationship.
Near 0: weak or no linear relationship.
Near -1: strong negative relationship.
'''

plt.figure(figsize=(6, 4))

sns.heatmap(
    df_correlation[["product_value", "freight_value"]].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Between Product Value and Freight Cost")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution analysis
# Checking how long orders take to reach customers.

query = """
SELECT
    DATEDIFF(
        order_delivered_customer_date,
        order_purchase_timestamp
    ) AS delivery_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL;
"""

df_delivery = pd.read_sql(query, connection)

df_delivery["delivery_days"].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df_delivery,
    x="delivery_days",
    bins=50,
    kde=True
)

plt.title("Distribution of Delivery Days")
plt.xlabel("Delivery Days")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate Analysis 
# Comparing delivery days, order value, and order status together.

query = """
SELECT
o.order_id,
o.order_status,
DATEDIFF(
        o.order_delivered_customer_date,
        o.order_purchase_timestamp
) AS delivery_days,
SUM(oi.price + oi.freight_value) AS total_order_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY o.order_id,
o.order_status,
o.order_delivered_customer_date,
o.order_purchase_timestamp;
"""

df_multivariate = pd.read_sql(query, connection)
df_multivariate.head()

In [ ]:
plt.figure(figsize=(12, 6))

sns.scatterplot(
    data=df_multivariate,
    x="delivery_days",
    y="total_order_value",
    hue="order_status",
    alpha=0.5
)

plt.title("Delivery Days vs Order Value by Order Status")
plt.xlabel("Delivery Days")
plt.ylabel("Total Order Value")
plt.tight_layout()
plt.show()